# L42 · 🎨 FFT 图形化 — 蝴蝶图 + 纯音 / 和弦 / 噪声的频谱形态对比

**目标**：用图直觉建立 DFT 矩阵、蝶形网络、频谱对称性的视觉记忆，
理解快速傅里叶变换（Fast Fourier Transform，FFT）「分而治之」背后的几何结构。

🔗 Aurora 连接：`aurora.audio.transforms.fft` · `aurora.audio.io.sine`

← **上一课**　[L41 · 加窗 FFT 完整流程](L41_fft_full.ipynb)

> 上节课学习了 **加窗 FFT 完整流程**：信号 → 加窗 → FFT → 幅度谱，一条管线跑通。  
> 本课将探讨 **FFT 图形化**。

## 核心直觉

DFT 本质是一次矩阵-向量乘法：把时域（time domain）信号投影到 N 个不同频率的复指数（complex exponential）基上。FFT 的魔法在于这个 N×N 矩阵有高度冗余——把它递归地折叠成两半，计算量从 `O(N²)` 降到 `O(N log N)`，每一层折叠对应图中的一组蝶形操作。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from aurora.audio.io import sine
from aurora.audio.transforms import fft

## 1. DFT 矩阵：N×N 酉矩阵（unitary matrix）

DFT 定义：`X[k] = sum_{n=0}^{N-1} x[n] * exp(-2*pi*i*k*n/N)`

写成矩阵形式，令 `W = exp(-2*pi*i/N)`，则

```
F[k, n] = W^(k*n),   k, n = 0, 1, ..., N-1
```

`F` 是 N×N **酉矩阵**（`F @ conj(F).T = N * I`），每一行是一个频率 k 的复指数，实部和虚部分别是余弦/正弦波。
热力图的行是频率 bin，列是时间采样点；颜色编码实部（余弦）。

In [ ]:
N = 16
n = np.arange(N)
k = np.arange(N)
# DFT 矩阵，形状 (N, N)
W = np.exp(-2j * np.pi * np.outer(k, n) / N)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
im0 = axes[0].imshow(W.real, cmap='RdBu_r', aspect='auto',
                     vmin=-1, vmax=1)
axes[0].set_title(f'DFT 矩阵实部（余弦）  N={N}')
axes[0].set_xlabel('时间采样点 n')
axes[0].set_ylabel('频率 bin k')
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(W.imag, cmap='RdBu_r', aspect='auto',
                     vmin=-1, vmax=1)
axes[1].set_title(f'DFT 矩阵虚部（正弦）  N={N}')
axes[1].set_xlabel('时间采样点 n')
axes[1].set_ylabel('频率 bin k')
plt.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.show()

# 验证酉性：F @ conj(F).T = N * I
err = np.max(np.abs(W @ W.conj().T - N * np.eye(N)))
print(f'酉性误差 max|F·F†  - N·I| = {err:.2e}  ✅' if err < 1e-10
      else f'酉性误差 {err:.2e}  ❌')

## 2. 蝶形网络：分而治之

Cooley-Tukey FFT 把长度 N 的 DFT 拆成两个 N/2 的 DFT：

```
X[k]     = E[k] + W^k * O[k]
X[k+N/2] = E[k] - W^k * O[k]
```

其中 `E[k]` 是偶数索引输入的 N/2 点 DFT，`O[k]` 是奇数索引的，
`W^k = exp(-2*pi*i*k/N)` 称为**旋转因子（twiddle factor）**。

N=8 时共有 `log2(8) = 3` 层，每层 4 个蝶形单元。
下图用节点-连线方式画出数据流，颜色标识旋转因子的相位。

In [ ]:
# 画 N=8 蝶形网络示意图（3 层，每层 4 个蝶形）
N = 8
stages = int(np.log2(N))  # 3

fig, ax = plt.subplots(figsize=(10, 5))
ax.set_xlim(-0.3, stages + 0.3)
ax.set_ylim(-0.5, N - 0.5)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('N=8 蝶形网络（3 层 × 4 蝶形）', fontsize=13)

# bit-reversal 置换顺序（输入顺序）
bit_rev = [int(f'{i:03b}'[::-1], 2) for i in range(N)]

# 节点 y 坐标（输入侧按 bit-reversal 排列）
y_pos = {i: bit_rev[i] for i in range(N)}
# 从第0层到第3层均匀铺开
colors = plt.cm.plasma(np.linspace(0.15, 0.85, stages))

def node_xy(stage, wire):
    return stage, wire

# 画输入标签（bit-reversal 后的顺序）
for i in range(N):
    ax.text(-0.15, bit_rev[i], f'x[{i}]', va='center', ha='right', fontsize=8)

# 逐层画蝶形
# current_y = list(range(N))  # 预留：追踪 wire 的 y 坐标（当前未使用）
for s in range(stages):
    half = 2 ** s          # 蝶形跨度
    group = 2 * half       # 组大小
    n_butterflies = N // group  # 每层蝶形组数
    col_in  = s
    col_out = s + 1
    for g in range(n_butterflies):
        for b in range(half):
            top = g * group + b
            bot = top + half
            yt, yb = top, bot
            # 旋转因子相位（0..0.5 映射到透明度，表达相位变化）
            phase = b / group  # 0..0.5
            alpha_cross = 0.4 + 0.5 * phase  # 相位越大斜线越明显
            c = colors[s]
            # 直通线（上）
            ax.annotate('', xy=(col_out, yt), xytext=(col_in, yt),
                        arrowprops=dict(arrowstyle='->', color=c, lw=1.4))
            # 直通线（下）
            ax.annotate('', xy=(col_out, yb), xytext=(col_in, yb),
                        arrowprops=dict(arrowstyle='->', color=c, lw=1.4))
            # 交叉线 top→bot（对应 +W^k 分支）
            ax.plot([col_in, col_out], [yt, yb], color=c, lw=0.8, ls='--', alpha=alpha_cross)
            # 交叉线 bot→top（对应 -W^k 分支，完整蝴蝶形结构）
            ax.plot([col_in, col_out], [yb, yt], color=c, lw=0.8, ls=':', alpha=alpha_cross)
            # 旋转因子标注（每个蝶形均标注，指数 = b * N // group）
            mid_x = (col_in + col_out) / 2
            mid_y = (yt + yb) / 2
            exp_k = b * N // group  # 正确的旋转因子指数
            ax.text(mid_x, mid_y + 0.15,
                    f'W^({N})^{{{exp_k}}}', fontsize=5.5, color=c,
                    ha='center', va='bottom')

# 输出标签
for i in range(N):
    ax.text(stages + 0.15, i, f'X[{i}]', va='center', ha='left', fontsize=8)

# 层标签
for s in range(stages):
    ax.text(s + 0.5, N - 0.1, f'第{s+1}层', ha='center', fontsize=9,
            color=colors[s], fontweight='bold')

plt.tight_layout()
plt.show()
print(f'N={N}，共 {stages} 层，每层 {N//2} 个蝶形，总操作数 = {N//2 * stages}（vs 朴素 DFT {N*N}）')
print('蝶形操作：X[top] = E + W^k·O（--线），X[bot] = E − W^k·O（:线）')


## 3. 频谱对称：实信号的镜像结构

当输入 `x[n]` 是**实数**时，DFT 满足共轭对称（conjugate symmetry）：

```
X[N-k] = conj(X[k]),   k = 1, 2, ..., N/2-1
```

因此幅度谱（magnitude spectrum） `|X[k]|` 关于 `k = N/2` 对称，只有前半段（`k = 0..N/2`）包含独立信息。
可视化中你会看到：左半段和右半段幅度是镜像；相位则反号。
Aurora 的 `fft` 输出全 N 个 bin，使用时取 `[:N//2+1]` 即可。

In [ ]:
# 用 aurora.audio.transforms.fft 演示共轭对称
sr = 8000
duration = 0.016  # 16 ms → N=128 采样（2^7，aurora FFT 需要 2 的幂）
x = sine(440.0, sample_rate=sr, duration=duration)  # 实数信号
N = len(x)

X = fft(x)          # aurora FFT，返回复数谱
mag   = np.abs(X)
phase = np.angle(X)

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
bins = np.arange(N)

axes[0].stem(bins, mag, markerfmt='C0o', linefmt='C0-', basefmt='k-')
axes[0].axvline(N//2, color='red', ls='--', lw=1, label=f'k=N/2={N//2}')
axes[0].set_ylabel('幅度 |X[k]|')
axes[0].set_title(f'440 Hz 实信号的幅度谱（N={N}）——关于 k=N/2 对称')
axes[0].legend()

axes[1].stem(bins, phase, markerfmt='C1o', linefmt='C1-', basefmt='k-')
axes[1].axvline(N//2, color='red', ls='--', lw=1)
axes[1].set_ylabel('相位 ∠X[k]  (rad)')
axes[1].set_xlabel('频率 bin k')
axes[1].set_title('相位谱——右半段是左半段的镜像反号')

plt.tight_layout()
plt.show()

# 数值验证：X[N-k] = conj(X[k])
err = np.max(np.abs(X[1:N//2] - np.conj(X[N-1:N//2:-1])))
print(f'共轭对称误差 max|X[N-k] - conj(X[k])| = {err:.2e}',
      '✅' if err < 1e-10 else '❌')

## 🏋️ 练习：手工实现 N=8 两级 Cooley-Tukey FFT

**任务**：不调用 `fft` 函数，用 NumPy 手工实现长度为 8 的迭代式 Cooley-Tukey FFT，
并与 `aurora.audio.transforms.fft` 的输出进行数值比较。

**步骤提示**：
1. 对长度 8 的输入按位逆序（bit-reversal permutation）重排：`bit_rev = [int(f'{i:03b}'[::-1], 2) for i in range(8)]`
2. 第 1 层（`half=1`）：相邻两元素做蝶形：`new_top = x[top] + W^0 * x[bot]`
3. 第 2 层（`half=2`）：跨度 2 的元素，旋转因子 `W^k = exp(-2πi·k/8)`
4. 第 3 层（`half=4`）：跨度 4 的元素，旋转因子同上
5. 对比 `aurora.audio.transforms.fft(x)` 结果，误差应低于 `1e-10`

**Aurora 连接**：`aurora.audio.transforms.fft` 就是这套算法的生产实现，完成后对比源码看看有何差异。


In [ ]:
def my_fft_n8(x):
    """手工实现 N=8 的迭代式 Cooley-Tukey FFT（不调用任何 FFT 库）。

    参数：x — 长度为 8 的实数或复数 numpy 数组
    返回：长度为 8 的复数频谱数组
    """
    N = 8
    assert len(x) == N, f"需要 N=8 输入，实际 N={len(x)}"

    # TODO: Step 1 — bit-reversal 置换
    # bit_rev = [int(f'{i:03b}'[::-1], 2) for i in range(N)]
    # X = np.array([x[bit_rev[i]] for i in range(N)], dtype=complex)
    raise NotImplementedError("请实现 bit-reversal 置换和蝶形迭代")

    # TODO: Step 2 — 三层蝶形迭代
    # for s in range(3):
    #     half = 2**s
    #     group = 2 * half
    #     for g in range(N // group):
    #         for b in range(half):
    #             top = g * group + b
    #             bot = top + half
    #             W_k = np.exp(-2j * np.pi * b / group)
    #             t = X[top] + W_k * X[bot]
    #             X[bot] = X[top] - W_k * X[bot]
    #             X[top] = t

    return X


# ── 验证（带 try/except 防呆：未实现时不崩溃）──
try:
    np.random.seed(42)
    x_test = np.random.randn(8)
    result_manual = my_fft_n8(x_test)
    result_aurora = fft(x_test)

    err = np.max(np.abs(result_manual - result_aurora))
    assert err < 1e-10, f"误差过大：{err:.2e}（期望 < 1e-10）"
    print(f"✅ 验证通过：手工 FFT 与 aurora.fft 最大误差 = {err:.2e}")
    print(f"手工结果 X[0..3] = {result_manual[:4]}")
except NotImplementedError:
    print("⚠️  练习未实现，请填写 my_fft_n8 函数后重新运行")
except AssertionError as e:
    print(f"❌ 验证失败：{e}")


## 参数实验：双频叠加信号（纯音叠加 = 和弦雏形）

**信号参数**：`sr=8000`，`duration=0.032`（32 ms，N=256，2 的幂次），
叠加 `f1=440 Hz`（bin ≈ 14）和 `f2=880 Hz`（bin ≈ 28）。

> **注意**：aurora FFT 要求输入长度为 2 的幂次。N=400 不是 2 的幂次，
> 若需要处理任意长度信号，应先用 zero-padding 补到下一个 2 的幂次。

**预期现象**：
- 幅度谱出现两个对称峰，峰位 bin 约为 `round(f * N / sr)`
- 880 Hz 的 bin 位置恰好是 440 Hz 的两倍（bin 28 = 2 × bin 14，八度关系）
- 超出 `N/2=128` 的镜像峰对称出现在高频侧（共轭对称性）


In [ ]:
sr       = 8000
duration = 0.032   # 32 ms → N=256（2^8）
f1, f2   = 440.0, 880.0

x1 = sine(f1, sample_rate=sr, duration=duration)
x2 = sine(f2, sample_rate=sr, duration=duration)
x  = x1 + x2
N  = len(x)

X   = fft(x)
mag = np.abs(X)

# 理论峰值 bin
bin1 = round(f1 * N / sr)
bin2 = round(f2 * N / sr)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(mag, color='steelblue', lw=1.2)
ax.axvline(bin1, color='C1', ls='--', label=f'{f1:.0f} Hz  bin={bin1}')
ax.axvline(N - bin1, color='C1', ls=':', alpha=0.5, label=f'镜像 bin={N-bin1}')
ax.axvline(bin2, color='C2', ls='--', label=f'{f2:.0f} Hz  bin={bin2}')
ax.axvline(N - bin2, color='C2', ls=':', alpha=0.5, label=f'镜像 bin={N-bin2}')
ax.axvline(N//2, color='red', ls='-', lw=0.8, alpha=0.4, label=f'N/2={N//2}')
ax.set_xlabel('频率 bin k')
ax.set_ylabel('幅度 |X[k]|')
ax.set_title(f'440 Hz + 880 Hz 叠加信号频谱  (sr={sr}, N={N})')
ax.legend()
plt.tight_layout()
plt.show()

print(f'440 Hz → bin {bin1}  （理论：{f1*N/sr:.2f}）')
print(f'880 Hz → bin {bin2}  （理论：{f2*N/sr:.2f}）')
print(f'两峰 bin 比值：{bin2}/{bin1} = {bin2/bin1:.2f}  （应≈2.0，八度）')

## 4. 频谱形态对比：纯音 / 和弦 / 白噪声

标题承诺的三类信号在这里汇合——直观对比它们的频谱形态：

| 信号类型 | 频谱特征 | 直觉解释 |
|---|---|---|
| **纯音**（单频正弦） | 单一尖峰 + 镜像峰 | 能量集中在一个频率 |
| **和弦**（多频叠加） | 多个离散峰 | 能量分布在几个频率 |
| **白噪声** | 宽带平坦响应 | 能量均匀分布在所有频率 |


In [ ]:
sr_cmp  = 8000
dur_cmp = 0.032  # 32 ms → N=256

# ── 纯音：440 Hz 单频正弦 ──
x_pure  = sine(440.0, sample_rate=sr_cmp, duration=dur_cmp)

# ── 和弦：440 + 660 + 880 Hz 三音叠加 ──
x_chord = (sine(440.0, sample_rate=sr_cmp, duration=dur_cmp) +
           sine(660.0, sample_rate=sr_cmp, duration=dur_cmp) +
           sine(880.0, sample_rate=sr_cmp, duration=dur_cmp))

# ── 白噪声：np.random.randn，理论上所有频率能量相等 ──
np.random.seed(0)
N_cmp   = len(x_pure)
x_noise = np.random.randn(N_cmp).astype(float)

# 计算各信号的幅度谱（只取前 N//2+1 个正频率 bin）
def half_spectrum(x):
    X = fft(x)
    return np.abs(X[:len(x)//2 + 1])

mag_pure  = half_spectrum(x_pure)
mag_chord = half_spectrum(x_chord)
mag_noise = half_spectrum(x_noise)

freqs = np.arange(len(mag_pure)) * sr_cmp / N_cmp

fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)

axes[0].plot(freqs, mag_pure, color='C0', lw=1.4)
axes[0].set_title('纯音（440 Hz）— 单一尖峰', fontsize=11)
axes[0].set_ylabel('幅度 |X[k]|')

axes[1].plot(freqs, mag_chord, color='C2', lw=1.4)
axes[1].set_title('和弦（440 + 660 + 880 Hz）— 三个离散峰', fontsize=11)
axes[1].set_ylabel('幅度 |X[k]|')

axes[2].plot(freqs, mag_noise, color='C3', lw=1.0, alpha=0.8)
axes[2].set_title('白噪声（randn）— 宽带平坦（随机起伏）', fontsize=11)
axes[2].set_ylabel('幅度 |X[k]|')
axes[2].set_xlabel('频率 (Hz)')

for ax in axes:
    ax.set_xlim(0, sr_cmp / 2)

plt.suptitle('频谱形态对比：纯音 / 和弦 / 白噪声', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# 数值验证：噪声频谱均值应接近其平均幅度（平坦性粗检验）
noise_std_ratio = mag_noise.std() / mag_noise.mean()
print(f'纯音   幅度谱峰值 bin = {np.argmax(mag_pure)} ({freqs[np.argmax(mag_pure)]:.0f} Hz)')
print(f'和弦   幅度谱前3峰 bin = {np.argsort(mag_chord)[-3:][::-1]} 对应 Hz ≈ {freqs[np.argsort(mag_chord)[-3:][::-1]].astype(int)}')
print(f'白噪声 幅度谱变异系数 = {noise_std_ratio:.2f}（值越小越平坦，白噪声理论≈1/√2）')


## 本课收束

`fft`（来自 `aurora.audio.transforms`）把 N 点实信号映射为 N 个复数 bin，
通过 DFT 矩阵可视化可以看到每一行是一个复指数基，蝶形网络则揭示了
`O(N log N)` 的递归折叠结构，共轭对称保证实信号只需保留前 `N//2+1` 个 bin。
下节 `L43_stft.ipynb` 将用滑动窗口把 FFT 扩展到时频谱（STFT），
看频率随时间演变的全貌。

In [ ]:
# ── 独立数学断言：验证 FFT 核心性质（无需学生实现）──────────────────────────
import numpy as np

np.random.seed(7)
x = np.random.randn(16)
N = len(x)
X = np.fft.fft(x)

# 1. DFT 矩阵酉性：F @ conj(F).T = N * I
n_arr = np.arange(N)
F = np.exp(-2j * np.pi * np.outer(n_arr, n_arr) / N)
err_unitary = np.max(np.abs(F @ F.conj().T - N * np.eye(N)))
assert err_unitary < 1e-10, f"酉性误差过大：{err_unitary:.2e}"
print(f"1 ✅  DFT 矩阵酉性：max|F·F†-N·I| = {err_unitary:.2e}")

# 2. 共轭对称：X[N-k] = conj(X[k])，k=1..N/2-1
err_sym = np.max(np.abs(X[1:N//2] - np.conj(X[N-1:N//2:-1])))
assert err_sym < 1e-10, f"共轭对称误差：{err_sym:.2e}"
print(f"2 ✅  共轭对称：max|X[N-k]-X[k]*| = {err_sym:.2e}")

# 3. Parseval 定理：Σ|x|² = (1/N)·Σ|X|²
energy_time = np.sum(np.abs(x)**2)
energy_freq = np.sum(np.abs(X)**2) / N
assert np.isclose(energy_time, energy_freq, rtol=1e-10)
print(f"3 ✅  Parseval：Σ|x|²={energy_time:.6f}，(1/N)Σ|X|²={energy_freq:.6f}")

# 4. 线性：FFT(ax+by) = a·FFT(x) + b·FFT(y)
y = np.random.randn(N); a, b = 2.5, -1.3
assert np.allclose(np.fft.fft(a*x + b*y), a*np.fft.fft(x) + b*np.fft.fft(y), atol=1e-10)
print(f"4 ✅  线性：FFT(ax+by) = a·FFT(x)+b·FFT(y)")

# 5. DC 分量：X[0] = Σ x[n]
assert np.isclose(X[0], np.sum(x), atol=1e-10)
print(f"5 ✅  DC 分量：X[0]={X[0]:.6f}，Σx={np.sum(x):.6f}")


---

→ **下一课**　[L43 · STFT 原理](L43_stft.ipynb)

> 下节课将学习 **STFT 原理**：短时傅里叶变换：给信号加时间戳，时频分辨率 tradeoff。